# 06c — KNN — SCAFFOLD

**Milestone 3 · Task 7.** Build the KNN classifier and record its
performance against the 54.7% majority-class floor.

K-Nearest Neighbors has no training phase. It memorises the training set,
and to classify a new campaign it finds the `k` most similar ones by
Euclidean distance and takes a majority vote.

**Scaling is mandatory** - unscaled, `log_goal` would dominate the
distance calculation and the binary flags would be invisible.
`make_models()` handles this.

**Expect this model to underperform, and do not treat that as a bug.**
After one-hot encoding the matrix has ~137 columns, and in high
dimensions distances concentrate: the nearest and farthest neighbours end
up almost equidistant, so "nearest" stops meaning "similar". One-hot
encoding makes it worse, since 49 subcategory columns each contribute to
distance while only one is ever 1. The underperformance is the finding.

Each notebook writes its own CSV to `reports/`, and Task 8 concatenates all three into one
comparison table.

**Two rules so the results line up:**

1. Build `X` and `y` with `build_model_matrix()` — never by hand.
2. Report through `evaluate()` — never write your own metric code. It returns
   a one-row DataFrame so the rows from all three notebooks stack cleanly.

Do not edit anything in `src/`.

## Setup

Run this first. On Colab it clones the repo; locally it does nothing.

In [1]:
# Works both locally and on Google Colab.
# On Colab this clones the repo; locally it does nothing.
import sys, os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "Global-Financial-Firm-1B-kickstarter-crowdfunding-recommendation-engine"
    GITHUB_URL = f"https://github.com/Break-Through-Tech/{REPO}.git"
    if not os.path.exists(REPO):
        os.system(f"git clone -q {GITHUB_URL}")
    os.chdir(f"{REPO}/notebooks")

sys.path.insert(0, "..")
print("Colab" if IN_COLAB else "local", "| cwd:", Path.cwd().name)

Colab | cwd: notebooks


In [2]:
import pandas as pd
import numpy as np

from src.features import build_model_matrix
from src.modeling import make_models, align_columns, random_split, chronological_split
from src.evaluation import evaluate, majority_class_baseline, confusion_frame

pd.set_option("display.width", 220)
EXTRA = ["log_city_project_count", "log_goal_vs_subcat_median"]

# The parquet splits are gitignored, so they will not exist on a fresh clone.
# Rebuild them if missing -- about 40 seconds.
if not Path("../data/m3_random_train.parquet").exists():
    print("Splits not found. Rebuilding from the raw data ...")
    from src.loading import load_raw
    from src.cleaning import clean
    from src.features import build_features, add_location_frequency, add_relative_goal

    feat = build_features(clean(load_raw("../data/DSI_kickstarterscrape_dataset_csv.zip"),
                                verbose=False)[0])

    def _prep(tr, te):
        tr, city = add_location_frequency(tr)
        tr, goal = add_relative_goal(tr)
        te, _ = add_location_frequency(te, reference=city)
        te, _ = add_relative_goal(te, reference=goal)
        return tr, te

    for (a, b), names in [(_prep(*random_split(feat)), ("m3_random_train", "m3_random_test")),
                          (_prep(*chronological_split(feat)), ("m3_chrono_train", "m3_chrono_test"))]:
        a.to_parquet(f"../data/{names[0]}.parquet", index=False)
        b.to_parquet(f"../data/{names[1]}.parquet", index=False)
    print("Splits written.")

SPLITS = {
    "random": (pd.read_parquet("../data/m3_random_train.parquet"),
               pd.read_parquet("../data/m3_random_test.parquet")),
    "chronological": (pd.read_parquet("../data/m3_chrono_train.parquet"),
                      pd.read_parquet("../data/m3_chrono_test.parquet")),
}
for n, (tr, te) in SPLITS.items():
    print(f"{n:15s} train {len(tr):,}  test {len(te):,}")

Splits not found. Rebuilding from the raw data ...
Splits written.
random          train 30,702  test 7,676
chronological   train 30,702  test 7,676


## The floor

Predicting the majority class for every row. Any model that fails to beat this has learned nothing, which is why reporting accuracy without this number is misleading.

Note the recall of 1.0: always guessing "successful" catches every success and is wrong about every failure. The floor still scores 0.71 on F1, so F1 alone is not enough either.

In [3]:
tr, te = SPLITS["random"]
floor = majority_class_baseline(tr["success"], te["success"])
floor.round(4)

,model,feature_set,split,accuracy,precision,recall,f1,roc_auc
0,Majority class,—,—,0.5473,0.5473,1.0,0.7074,0.5


## Your task

Fit **KNN** on all four combinations:

| feature_set | split |
|---|---|
| `launch_only` | `random` |
| `launch_only` | `chronological` |
| `plus_campaign` | `random` |
| `plus_campaign` | `chronological` |

`launch_only` is the primary feature set — it excludes `updates`, which is
unavailable at campaign launch. `plus_campaign` adds it back so the cost of
that exclusion can be measured in Task 8 rather than argued about.

**Expect roughly 65–72% on `launch_only`, not the 80% in the brief.** That is
the correct result, not a mistake: the benchmark was produced with `updates`
included.

### The pattern for one combination

```python
train, test = SPLITS["random"]
X_train, y_train = build_model_matrix(train, "launch_only", extra_numeric=EXTRA)
X_test,  y_test  = build_model_matrix(test,  "launch_only", extra_numeric=EXTRA)
X_test = align_columns(X_train, X_test)   # one-hot columns must match exactly

model = make_models()["KNN"]
model.fit(X_train, y_train)

evaluate(model, X_test, y_test,
         label="KNN", feature_set="launch_only", split="random")
```

Loop that over both feature sets and both splits, collect the rows, and
`pd.concat` them.

In [4]:
results = []

for split_name, (train, test) in SPLITS.items():
    for fs in ["launch_only", "plus_campaign"]:
        X_train, y_train = build_model_matrix(train, fs, extra_numeric=EXTRA)
        X_test,  y_test  = build_model_matrix(test,  fs, extra_numeric=EXTRA)
        X_test = align_columns(X_train, X_test)   # one-hot columns must match exactly
        #       build_model_matrix(), then align_columns() the test matrix

        # TODO: model = make_models()["KNN"]  then model.fit(...)
        model = make_models()["KNN"]
        model.fit(X_train, y_train)
        results.append(evaluate(model, X_test, y_test,
                             label="KNN", feature_set=fs, split=split_name))
        print(f"done: {fs:14s} | {split_name}")


res = pd.concat(results, ignore_index=True)
res.round(4)

done: launch_only    | random
done: plus_campaign  | random
done: launch_only    | chronological
done: plus_campaign  | chronological


,model,feature_set,split,accuracy,precision,recall,f1,roc_auc
0,KNN,launch_only,random,0.6528,0.6453,0.8119,0.7191,0.7026
1,KNN,plus_campaign,random,0.6855,0.6746,0.8217,0.7409,0.7413
2,KNN,launch_only,chronological,0.6567,0.6508,0.7730,0.7067,0.7138
3,KNN,plus_campaign,chronological,0.6794,0.6765,0.7679,0.7193,0.7410


## Inspect the results

Once the loop runs, look at the primary configuration — `launch_only` on the `random` split. That is the one that answers the project's actual question.

In [6]:
# TODO: filter res to feature_set == "launch_only" and split == "random"
primary = res[(res["feature_set"] == "launch_only") & (res["split"] == "random")].iloc[0]
# primary.round(4)

# TODO: compare its accuracy against the floor above
knn_acc   = primary["accuracy"]
floor_acc = floor["accuracy"].iloc[0]
lift      = knn_acc - floor_acc

# TODO: print the lift over the floor
print(f"KNN accuracy (launch_only / random): {knn_acc:.1%}")
print(f"Majority-class floor:                {floor_acc:.1%}")
print(f"Lift over the floor:                 {lift * 100:+.1f} percentage points")

KNN accuracy (launch_only / random): 65.3%
Majority-class floor:                54.7%
Lift over the floor:                 +10.6 percentage points


## Confusion matrix

`confusion_frame(model, X_test, y_test)` returns a labelled matrix. The two off-diagonal cells are different harms:

- **False positive** — a creator is told to launch, and the campaign fails
- **False negative** — a creator is discouraged from a campaign that would have succeeded

Which matters more is a question Milestone 4 has to answer.

In [7]:
# TODO: refit on launch_only / random and show confusion_frame(...)
train, test = SPLITS["random"]
X_train, y_train = build_model_matrix(train, "launch_only", extra_numeric=EXTRA)
X_test,  y_test  = build_model_matrix(test,  "launch_only", extra_numeric=EXTRA)
X_test = align_columns(X_train, X_test)

knn_primary = make_models()["KNN"]
knn_primary.fit(X_train, y_train)

confusion_frame(knn_primary, X_test, y_test)

,predicted: failed,predicted: successful
actual: failed,1600,1875
actual: successful,790,3411


## Save your results

Task 8 reads this file. **Use exactly this filename** or the comparison will not pick it up.

In [8]:
# TODO: uncomment once `res` exists
res.to_csv("../reports/m3_task7_knn.csv", index=False)
print("Wrote ../reports/m3_task7_knn.csv")

Wrote ../reports/m3_task7_knn.csv


## Write-up

Fill this in when the numbers are available.

- **Accuracy on `launch_only` / `random`:** 65.3% (floor is 54.7%)
- **Lift over the floor:** +10.6 pp
- **What `updates` is worth:** +3.3 points of accuracy on the random split (65.3% → 68.6%) and +2.3 on chronological (65.7% → 67.9%). ROC AUC goes up about 0.03–0.04 in both cases. (launch_only vs plus_campaign)
- **Chronological vs random:** basically the same. launch_only is 65.7% vs 65.3%, and plus_campaign is 67.9% vs 68.6%. So the model holds up when it has to predict future campaigns from past ones, which is a good sign.

**NOTE**: It leans toward predicting “success.” Recall is about 0.81 but precision is only about 0.65, which means it catches most successful campaigns but also flags a lot of failures as successes. In the confusion matrix, that’ll show up as more false positives (creators told to launch whose campaigns then fail) than false negatives.

### Things to watch with this model

- `n_neighbors` is the whole model. Small values are noisy; large values
  smooth toward the majority class.
- Prediction is slow because every prediction scans the training set.
  Relevant for Milestone 4 if this were ever the final model.
- It should still beat the 54.7% floor. If it does not, something is wrong
  with the matrix, not with KNN.